# Portfolio Drift Monitor - Exploratory Data Analysis

**Stage 08.** This notebook is the reference the later stages cite. It answers three
questions about the monitor's own data:

1. What shape is it in, and does anything need cleaning that Stage 06 did not catch?
2. What is the drift actually doing over time, and does that change what the monitor
   means?
3. What should Stage 09 build, and what must Stage 10b be careful about?

It runs standalone from the cleaned prices the pipeline saves, so it can be re-run
without re-downloading. The pipeline itself carries a short Stage 08 section that
re-runs the checks and saves the evidence; the reasoning lives here.

In [ ]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                               # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))                # so `from src....` imports work
print('working from:', ROOT.name)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import eda
from src.utils import TARGET_WEIGHTS, AMBER_PP, RED_PP, drift_pp, flag_drift, read_df

sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

TICKERS = list(TARGET_WEIGHTS)
RAW = Path('data/raw')
PROC = Path('data/processed')
IMG = Path('reports/images')
IMG.mkdir(parents=True, exist_ok=True)

print('targets   :', TARGET_WEIGHTS)
print('thresholds: amber %.1fpp, red %.1fpp' % (AMBER_PP, RED_PP))

## 1. Load the cleaned prices

Read the most recent cleaned table the pipeline saved. If none exists yet, fall back to
the most recent raw pull and clean it here, so this notebook never depends on the
pipeline having been run first.

In [ ]:
clean_files = sorted(PROC.glob('prices_clean_*.parquet'))

if clean_files:
    src_path = clean_files[-1]
    prices_clean = read_df(src_path)
else:
    from src import cleaning
    src_path = sorted(RAW.glob('api_yfinance_*.csv'))[-1]
    prices_clean, _ = cleaning.clean_prices(pd.read_csv(src_path), TICKERS)

prices_clean['date'] = pd.to_datetime(prices_clean['date'])
print('source:', src_path.name)
print('shape :', prices_clean.shape)
print('window:', prices_clean['date'].min().date(), '->', prices_clean['date'].max().date())
prices_clean.head()

In [ ]:
# Rebuild weights and drift from the cleaned prices, using the same convention as the
# pipeline: buy the target on day one and never rebalance, so drift is what the market
# does to a portfolio left alone.
wide = prices_clean.pivot(index='date', columns='ticker', values='close')[TICKERS]

targets = np.array([TARGET_WEIGHTS[t] for t in TICKERS])
shares = (1_000_000 * targets) / wide.iloc[0].to_numpy()
values = wide.to_numpy() * shares
weights_df = pd.DataFrame(values / values.sum(axis=1, keepdims=True),
                          index=wide.index, columns=TICKERS)
drift_df = pd.DataFrame(drift_pp(weights_df.to_numpy(), targets),
                        index=wide.index, columns=TICKERS)
rets = wide.pct_change(fill_method=None)

drift_long = (drift_df.reset_index()
                      .melt(id_vars='date', var_name='ticker', value_name='drift_pp'))
drift_long['target_weight'] = drift_long['ticker'].map(TARGET_WEIGHTS)
drift_long['flag'] = flag_drift(drift_long['drift_pp'])
drift_long['asset_class'] = np.where(drift_long['ticker'] == 'BND', 'bond', 'equity')

assert np.allclose(weights_df.iloc[0], targets), 'day 0 must sit exactly on target'
print(f'{len(wide)} trading days, {len(drift_long)} fund-days')
drift_long.head()

## 2. Structure, missingness, numeric profile

In [ ]:
summary = eda.eda_summary(drift_long)
print('shape    :', summary['shape'])
print('missing  :', {k: v for k, v in summary['missing'].items() if v} or 'none')
display(summary['numeric_profile'])

print()
print('per-fund return profile')
returns = eda.return_profile(prices_clean)
display(returns)

**What?** No missing values anywhere. Annualized volatility is 13.1% for VTI, 17.0% for
VXUS and 3.9% for BND. Excess kurtosis is 2.0 for VXUS, 1.1 for VTI and 0.07 for BND.

**So what?** Zero missingness is Stage 06 working: `drop_incomplete_days` removes any
date lacking a close for even one fund, because a weight computed from two funds out of
three is wrong rather than approximate. The cost is invisible here and worth naming, so
it appears in the assumptions below.

The kurtosis column is a retroactive check on Stage 07. Excess kurtosis is measured
against a normal distribution, where the value is 0. VXUS at 2.0 has visibly heavier
tails than normal, which is exactly the region a 3-sigma z-score cutoff assumes it knows.
That is why the IQR rule was made the primary detector and the z-score the cross-check,
and this profile is the evidence for it. BND at 0.07 is the one fund where a z-score
would behave.

**Now what?** Any per-fund statistic stays per-fund. A pooled standard deviation across a
bond fund and two equity funds describes neither.

## 3. Categorical profile

In [ ]:
for col, table in eda.categorical_profile(drift_long).items():
    print(f'--- {col} ---')
    print(table.to_string())
    print()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.countplot(data=drift_long, x='ticker', ax=axes[0], hue='ticker', legend=False)
axes[0].set_title('Fund-days per fund')
sns.countplot(data=drift_long, x='flag', order=['green', 'amber', 'red'],
              ax=axes[1], hue='flag', hue_order=['green', 'amber', 'red'], legend=False)
axes[1].set_title(f'Drift flag (amber {AMBER_PP}pp, red {RED_PP}pp)')
fig.tight_layout()
plt.show()

**What?** 251 fund-days each, exactly. `flag` is 100% green; amber and red do not appear.

**So what?** The balanced split is a passing test rather than a curiosity: Stage 06
guarantees it, so an imbalance would mean the cleaner had been skipped. The `flag`
column is the headline. Over 753 fund-days the alarm this project exists to raise has
never gone off, which Stage 07 documented. Section 6 works out why, and that turns out
to matter more than the count does.

**Now what?** `flag` is a reporting column, not a feature. Section 8's `flag_columns`
picks it up automatically so Stage 09 cannot forget.

## 4. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, t in zip(axes.flat, TICKERS):
    sns.histplot(rets[t].dropna() * 100, kde=True, bins=40, ax=ax)
    ax.set_title(f'{t} daily returns (%)')
    ax.set_xlabel('daily return (%)')
sns.boxplot(data=prices_clean.assign(ret=prices_clean.sort_values(['ticker', 'date'])
                                     .groupby('ticker')['close']
                                     .pct_change(fill_method=None) * 100),
            x='ticker', y='ret', ax=axes[1, 1], hue='ticker', legend=False)
axes[1, 1].set_title('Daily returns by fund (outliers)')
axes[1, 1].set_ylabel('daily return (%)')
fig.tight_layout()
fig.savefig(IMG / 'eda_return_distributions.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for t in TICKERS:
    sns.histplot(drift_df[t], bins=35, element='step', alpha=0.5, label=t, ax=axes[0])
axes[0].set_title('Drift distribution by fund')
axes[0].set_xlabel('drift (percentage points)')
axes[0].legend()

sns.boxplot(data=drift_long, x='ticker', y='drift_pp', ax=axes[1],
            hue='ticker', legend=False)
axes[1].axhline(AMBER_PP, color='orange', linestyle='--', label=f'amber {AMBER_PP}pp')
axes[1].axhline(-AMBER_PP, color='orange', linestyle='--')
axes[1].set_title('Drift against the amber line')
axes[1].set_ylabel('drift (pp)')
axes[1].legend(loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(IMG / 'eda_drift_distributions.png', dpi=110, bbox_inches='tight')
plt.show()

**What?** Returns are near-symmetric and centred on zero, with BND's spread roughly a
third of VTI's. The drift distributions are *not* centred on zero: VXUS sits mostly
positive, BND mostly negative, VTI straddles. The amber line is far above every box.

**So what?** A drift distribution that is not centred on zero is the most important thing
on this page. Drift oscillating around zero would be noise, and the right response would
be to leave it alone. Drift with a non-zero centre is a position that has been moving one
way, which is a different problem with a different fix. Section 6 settles which it is.

**Now what?** No log transform on returns. They are near-symmetric and contain negatives.

## 5. Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

sns.regplot(x=rets['VTI'] * 100, y=rets['VXUS'] * 100, ax=axes[0],
            scatter_kws={'alpha': 0.5, 's': 22}, line_kws={'color': 'crimson'})
axes[0].set_title('VTI vs VXUS daily returns')
axes[0].set_xlabel('VTI (%)'); axes[0].set_ylabel('VXUS (%)')

sns.regplot(x=rets['VTI'] * 100, y=rets['BND'] * 100, ax=axes[1],
            scatter_kws={'alpha': 0.5, 's': 22}, line_kws={'color': 'crimson'})
axes[1].set_title('VTI vs BND daily returns')
axes[1].set_xlabel('VTI (%)'); axes[1].set_ylabel('BND (%)')

sns.scatterplot(x=drift_df['VXUS'], y=drift_df['BND'], ax=axes[2], s=26, alpha=0.75,
                hue=drift_df.index.to_period('Q').astype(str))
axes[2].set_title('VXUS drift vs BND drift')
axes[2].set_xlabel('VXUS drift (pp)'); axes[2].set_ylabel('BND drift (pp)')
axes[2].legend(title='quarter', fontsize=7, title_fontsize=8)

fig.tight_layout()
fig.savefig(IMG / 'eda_relationships.png', dpi=110, bbox_inches='tight')
plt.show()

print('daily return correlations')
print(rets.corr().round(3).to_string())

**What?** VTI and VXUS returns correlate at 0.82, so one explains about 67% of the
other's daily variance. VTI and BND correlate at 0.34, VXUS and BND at 0.46. The third
panel shows the two drift series in a tight negative band, with the quarters laid out
along it in order rather than scattered through it.

**So what?** 0.82 is a warning for Stage 09: two features sharing two thirds of their
variance are close to one feature, and a linear model fed both produces coefficients
that flip sign on a slightly different sample. BND at 0.34 is genuinely carrying
different information, which is the whole argument for holding it.

The third panel is a trap, and section 7 names it. What is real in that panel is the
ordering of the quarters, which is a time effect rather than a relationship between two
funds.

**Now what?** Use one of VTI / VXUS, or replace the pair with a spread.

## 6. The time axis, and what the drift does along it

This is the section Stage 09 and Stage 10b depend on. Two questions: is the axis sound,
and what is the series doing along it.

In [ ]:
daily = wide.reset_index()[['date']]              # one row per trading day

for label, frame, freq in [('long frame, business grid', drift_long, 'B'),
                           ('daily axis, CALENDAR grid', daily, 'D'),
                           ('daily axis, BUSINESS grid', daily, 'B')]:
    rep = eda.time_axis_report(frame, freq=freq)
    print(f'--- {label} ---')
    for k, v in rep.items():
        if k != 'missing_periods':
            print(f'  {k:<20} {v}')
    print()

bus = eda.time_axis_report(daily, freq='B')
print('business days genuinely absent (all US market holidays):')
for d in bus['missing_periods']:
    print('  ', d.date())

gaps = daily['date'].diff().dt.days.dropna().astype(int)
print()
print('calendar days between consecutive rows')
print(gaps.value_counts().sort_index().to_string())

**What?** No duplicates on the daily axis, rows sorted, and 114 calendar days missing
against only 10 business days. All ten are named US market holidays. Gaps between rows
are 1 day (195 times), 3 days (45, weekends), 4 days (7, holiday weekends) and 2 days (3).

The long frame reports 502 duplicate dates, which is correct: three funds share every
date. That is why the axis check runs on the one-row-per-date frame.

**So what?** No data-quality problem, but a units problem that would silently corrupt
Stage 09. The axis is business-daily. A 7-row rolling mean spans 9 calendar days on
average and 11 across a holiday weekend. `shift(1)` means the previous trading day, which
is three calendar days back every Monday. Reindexing onto `freq='D'` to "make the axis
regular" would insert 114 empty rows and dilute every rolling statistic, with no warning.

**Now what?** Stage 09 builds lags and rolling windows on the existing business-day index
and labels them in trading days. It does not reindex to `'D'`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for t in TICKERS:
    axes[0].plot(drift_df.index, drift_df[t], linewidth=1.4, label=t)
axes[0].axhline(0, color='grey', linewidth=0.8)
axes[0].axhline(AMBER_PP, color='orange', linestyle='--', linewidth=1)
axes[0].axhline(-AMBER_PP, color='orange', linestyle='--', linewidth=1)
axes[0].axhline(RED_PP, color='firebrick', linestyle='--', linewidth=1)
axes[0].axhline(-RED_PP, color='firebrick', linestyle='--', linewidth=1)
axes[0].set_ylim(-6, 6)
axes[0].set_title('Drift from target, against the amber and red lines')
axes[0].set_ylabel('drift (pp)')
axes[0].legend(loc='upper left', ncol=3)

roll_vol = rets.rolling(21).std() * np.sqrt(252) * 100
for t in TICKERS:
    axes[1].plot(roll_vol.index, roll_vol[t], linewidth=1.4, label=t)
axes[1].set_title('21-trading-day rolling annualized volatility')
axes[1].set_ylabel('vol (%)'); axes[1].set_xlabel('date')
axes[1].legend(loc='upper left', ncol=3)

fig.tight_layout()
fig.savefig(IMG / 'eda_drift_over_time.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
trend = eda.drift_trend(drift_long, amber=AMBER_PP, red=RED_PP)
display(trend)

print('cumulative total return over the window (%)')
print(((wide.iloc[-1] / wide.iloc[0] - 1) * 100).round(2).to_string())

**What?** BND's drift correlates with time at -0.93, so 86% of its variation is a
straight line, and it is falling at 1.69pp per year. VXUS rises at 1.35pp per year with
an r-squared of 0.41. VTI is close to flat: 0.34pp per year, r-squared 0.03. Underneath,
the funds returned +19.2%, +22.6% and -2.0% over the window. Rolling volatility moves
between 6% and 21% for VTI and 8% and 29% for VXUS, in visible clusters.

**So what? This reframes the monitor.** There is no seasonality here and no level shift.
There is a trend, and in BND's case it is close to deterministic. Drift is not a
portfolio wobbling around its target and occasionally poking through a line. It is a
one-way ratchet driven by a persistent equity-over-bond return spread, and one year is
simply too short for the ratchet to reach 3pp.

Projecting the fitted rates forward *from today's drift*, and stating plainly that this
assumes the spread persists: BND is about 0.8 years from amber and 2.0 from red. VXUS is
about 1.4 from amber and 2.9 from red. VTI is years away from either.

That is a far more useful sentence than "the alarm has never fired". The alarm is not
broken and the threshold is not obviously wrong. The monitor has been shown one year of
a phenomenon that operates on a multi-year timescale, and on current rates the first
amber is inside the next reporting year.

**Now what, for Stage 09.** Features go on the *rate* of drift, not the level. A trended
level makes a lagged level almost as good a predictor of tomorrow as today's value is,
which adds nothing. The candidates, in order: the first difference of drift; a rolling
21-day and 63-day slope; the equity-minus-bond return spread, which is the mechanism
producing the drift; and 21-day realized volatility, since the vol plot shows real
clustering. Calendar features are not worth building, because there is no seasonality
here to catch.

**Now what, for Stage 10b.** Split chronologically, and expect it to look bad. Training
on the first 80% trains on the low-drift half and tests on the high-drift half, so the
test set is out of distribution by construction. A random split would score much better
and would be leakage, because a trend makes neighbouring days nearly identical. Report
the drift range of each side alongside the score so the comparison is honest.

## 7. Correlation: one real matrix and one structural one

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
sns.heatmap(rets.corr(), annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1,
            ax=axes[0], cbar=False)
axes[0].set_title('Daily RETURNS (real)')
sns.heatmap(drift_df.corr(), annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1,
            ax=axes[1], cbar=False)
axes[1].set_title('DRIFT (structural, do not interpret)')
fig.tight_layout()
fig.savefig(IMG / 'eda_correlation.png', dpi=110, bbox_inches='tight')
plt.show()

print('drift rows sum to (should be ~0 everywhere):',
      float(drift_df.sum(axis=1).abs().max()))

**What?** Returns: 0.82 between the equity funds, 0.34 and 0.46 against BND. Drift: every
pair negative, and the three drift columns sum to zero on every row to within floating
point.

**So what?** Only the left panel is data. The right panel is arithmetic. Weights sum to
one, so deviations from a fixed target sum to zero, so the columns are forced to move
against each other whatever the market does. Reading it as "the funds diversify each
other" would be a real error, and a heatmap makes that error easy because both panels
look equally like results. This is the reading's warning about correlation being a hint
rather than a claim, in its sharpest form: here it is not even a hint, it is a
restatement of the definition of a weight.

**Now what?** Only the returns matrix informs feature selection. The pair that matters is
VTI and VXUS. Everything involving BND is low enough to keep both sides.

## 8. Columns needing a decision before Stage 09

In [ ]:
flagged = eda.flag_columns(drift_long)
if flagged.empty:
    print('nothing flagged at the default thresholds')
else:
    print(flagged.to_string(index=False))

`flag` comes back as a dominant category at 100%. That is section 3's finding arriving
without anyone having to remember to look for it, which is the point of writing the check
as a function rather than as a cell.

## 9. Save the evidence

In [ ]:
import datetime as dt
stamp = dt.datetime.now().strftime('%Y%m%d-%H%M')

returns.to_csv(PROC / f'eda_return_profile_{stamp}.csv')
trend.to_csv(PROC / f'eda_drift_trend_{stamp}.csv')
flagged.to_csv(PROC / f'eda_flagged_columns_{stamp}.csv', index=False)

for p in sorted(PROC.glob(f'eda_*{stamp}.csv')):
    print(f'{p.name:<44} {p.stat().st_size:>6} bytes')
print()
for p in sorted(IMG.glob('eda_*.png')):
    print(f'{p.name:<44} {p.stat().st_size:>6} bytes')

## 10. Insights, assumptions, and what happens next

### Top 3 insights

**1. The drift is a trend, not noise, and the first amber is probably inside the next
reporting year.** BND's drift correlates with time at -0.93 and falls at 1.69pp per year;
VXUS rises at 1.35pp. Underneath is a persistent return spread: +19.2% and +22.6% for the
equity funds against -2.0% for the bond fund. So the thresholds are not mis-set and the
alarm is not broken. From today's drift, BND is roughly 0.8 years from amber and VXUS
roughly 1.4. Stage 07 could only report that nothing had fired; this dates it.

**2. The date axis is business-daily and clean, and treating it as calendar-daily would
quietly corrupt Stage 09.** 251 sorted rows, no duplicates, and the only ten absent
business days are named market holidays. But 114 calendar days are missing, so a 7-row
rolling window spans 9 calendar days on average, and reindexing to `freq='D'` would
insert 114 empty rows that dilute every rolling statistic without raising an error.

**3. Two of the three funds are close to one fund, and the drift correlation matrix is
not evidence of anything.** VTI and VXUS returns correlate at 0.82. Separately, the drift
correlations are negative on every pair for a purely structural reason: weights sum to
one, so drifts sum to zero.

### Assumptions and risks

| Assumption | Risk if wrong |
|---|---|
| One year of daily closes represents the behaviour worth monitoring | This was a calm, trending year. A volatile one would show spikes this analysis says are absent, and the trend projection would be swamped. |
| The linear projection of drift rates is indicative, not a forecast | It assumes the equity-over-bond spread persists. It will not persist indefinitely, and mean reversion pushes the amber dates out. Use it to size the problem, not to schedule a trade. |
| Prices are correct as delivered by yfinance | Stage 07 examined 24 IQR-flagged fund-days and found them genuine market moves, including five dates where two funds moved together. No prices were removed. Inherited here. |
| Dates with an incomplete set of closes were correctly dropped upstream | Stage 06 deletes rather than repairs those days, which is why missingness is zero here. The deletions are invisible in the cleaned file. |
| Percentage points are the right threshold unit | BND's worst drift is 17.7% of its 10% target; VTI's is 2.6% of its 60% target. A single pp rule is least sensitive where a given move matters most. Open in `docs/outliers.md`, still a decision for the principal. |

### What this means for the report Dana receives

Dana is a client-service associate who lives in Excel and will not run scripts. The
practical consequence of insight 1 is that a monitor that has printed "green" every day
for a year is about to start printing something else, and it should not be the first
time she sees an amber row that anyone explains what it means. The drift trend table
belongs in the report as a standing line, not just in this notebook.

The stretch test in the homework notebook is relevant here too: month-end sampling
reproduced the worst drift for VTI and VXUS exactly and understated BND by 0.064pp. If a
monthly cadence is operationally easier, one year of evidence says it costs almost
nothing. The caveat is that the evidence covers a trending year with no spikes.

### Next steps

**Cleaning:** nothing further. Complete, correctly typed, sorted, no duplicates.

**Stage 09 features**, in priority order: first difference of drift; rolling 21-day and
63-day drift slope; equity-minus-bond return spread; 21-day realized volatility per fund.
Drop one of VTI / VXUS or replace the pair with a spread. Skip calendar features. Build
everything on the business-day index, labelled in trading days.

**Stage 10b:** chronological split, with the drift range of each side reported next to
the score.